# NaN handling

Real time series have gaps: a failed sensor, a market halt, a missing tick. Every
screamer function declares how it treats a NaN on its input, and there are three
behaviours:

- **`ignore`**: the NaN is skipped and the output is NaN at that point, but the
  internal state is untouched, so the next value recovers. Used by the summary
  statistics (rolling, EW, bands, oscillators, filters).
- **`propagate`**: the NaN flows through the formula, so it lingers as long as it
  sits in the lookback, then clears. Used by positional functions (`Diff`, `Lag`,
  `Return`, and similar).
- **`nan-aware`**: the function exists to remove NaN. `FillNa` and `Ffill`.

Which function uses which policy, and the exact rules, are in
[NaN and warmup](../nan_and_warmup.md). This notebook shows all three on the same
series with a single gap.

In [ ]:
import numpy as np
import pandas as pd
from screamer import RollingMean, Diff, FillNa, Ffill, Dropna

x = np.array([1.0, 2.0, 3.0, np.nan, 5.0, 6.0])   # one gap, at t = 3

## The three policies on one gap

Running one representative function per policy on the same series lines the
behaviours up side by side. Watch the gap at `t = 3`, and how far the NaN spreads
after it.

In [ ]:
table = pd.DataFrame({
    "input": x,
    "ignore: RollingMean(3)": RollingMean(3)(x).round(3),
    "propagate: Diff(1)": Diff(1)(x),
    "nan-aware: FillNa(0)": FillNa(0.0)(x),
    "nan-aware: Ffill": Ffill()(x),
})
table.index.name = "t"
table

Reading the gap column and the rows after it:

- **ignore** (`RollingMean`) is NaN only at `t = 3`; the window skipped the gap,
  so `t = 4` is finite again.
- **propagate** (`Diff`) is NaN at `t = 3` and `t = 4`. `Diff(1)` compares each
  value with the previous one, so the gap spoils two outputs before it slides out
  at `t = 5`. Keeping the NaN visible is safer than silently differencing across
  the gap.
- **nan-aware** (`FillNa`, `Ffill`) has no NaN at all: the gap is replaced by a
  constant or by the last finite value. There is no backward fill, since that
  would need future data.

## Dropping gaps across streams: `Dropna`

The three policies act *within* a single series. The multi-stream layer adds
`Dropna`, which removes the events whose value is NaN entirely, giving a shorter,
gap-free stream. It changes the length of the stream (fewer events out than in)
and returns values first, as `(values, index)`. See
[Streams, values, and alignment](../multistream.md).

In [ ]:
idx = np.arange(x.size)
values, kept_idx = Dropna()(x, index=idx)

print("input       :", x)
print("kept values :", values)
print("kept index  :", kept_idx)
print(f"length {x.size} -> {values.size} (one gap removed)")